# RTE Concept-Direction Analysis with Interpretune

This notebook demonstrates concept-direction analysis for the RTE task using
Interpretune's analysis-op framework with the NNsight circuit-tracer backend
and real HuggingFace RTE data.

Key concepts demonstrated:
- Loading RTE data via RTEBoolqDataModule and test_dataloader
- Static concept direction computation from "Yes" vs "No" embedding groups
- Quality gate: projecting vocabulary onto the concept direction
- Validating direction alignment on real RTE batches
- Attribution graph generation and top-feature extraction on real examples
- Feature intervention on multiple real examples

In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
core_log_dir = None  # Analysis log directory; temp dir when unset
intervention_scale_factor = 30.0  # Multiplier for feature intervention amplitudes
n_top = 10  # Number of top features to extract
n_validation_batches = 3  # Number of dataloader batches to inspect

In [ ]:
# @title Imports { display-mode: "form" }
import gc

import torch

import interpretune as it
from it_examples import _ACTIVE_PATCHES  # noqa: F401
from it_examples.example_module_registry import MODULE_EXAMPLE_REGISTRY
from it_examples.utils.example_helpers import required_os_env
from it_examples.utils.nb_ui_utils import (
    display_top_features_comparison,
    display_topk_token_predictions,
)
from interpretune import ITSession, ITSessionConfig
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.analysis.ops.helpers import resolve_embedding_weight, resolve_tokenizer
from interpretune.config import AnalysisCfg, init_analysis_cfgs

In [ ]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to "/full/path/to/.env" to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

In [ ]:
# Load the Gemma-2 NNsight circuit-tracer demo configuration from the registry
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = MODULE_EXAMPLE_REGISTRY.get("gemma2.rte_demo.circuit_tracer_nnsight")

if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir

# Use a lighter notebook runtime than the parity-oriented registry defaults.
base_it_cfg.nnsight_cfg.torch_dtype = torch.bfloat16
base_it_cfg.circuit_tracer_cfg.batch_size = 32
base_it_cfg.circuit_tracer_cfg.max_feature_nodes = 1024
base_it_cfg.circuit_tracer_cfg.intervention_value_source = "top_feature_activation_values"
base_it_cfg.circuit_tracer_cfg.intervention_scale_factor = intervention_scale_factor
base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"

session_cfg = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)

it_session = ITSession(session_cfg)
it.it_init(**it_session)
module = it_session.module
datamodule = it_session.datamodule
tokenizer = resolve_tokenizer(module)

module.analysis_cfg = AnalysisCfg(target_op=it.compute_attribution_graph, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

test_dl = datamodule.test_dataloader()

print(f"Model loaded: {type(module.replacement_model).__name__}")
print("Backend: NNsight circuit-tracer")
print(f"Dataset: SuperGLUE RTE validation split (batch_size={test_dl.batch_size})")
print(f"Attribution runtime dtype: {base_it_cfg.nnsight_cfg.torch_dtype}")
print(f"Attribution batch_size: {base_it_cfg.circuit_tracer_cfg.batch_size}")
print(f"Attribution max_feature_nodes: {base_it_cfg.circuit_tracer_cfg.max_feature_nodes}")
print(f"Total batches available: {len(test_dl)}")

---
## Phase A — Concept Direction Quality Validation

We compute a static concept direction that separates entailment (Yes) from
non-entailment (No) in Gemma-2's embedding space. Then we validate that
direction on real RTE examples drawn from the dataloader.

In [ ]:
# Define concept groups: "Yes" (entailment) vs "No" (non-entailment)
concept_group_a = ["▁Yes"]
concept_group_b = ["▁No"]

cd_result = it.concept_direction(
    module,
    AnalysisBatch(
        concept_group_a=concept_group_a,
        concept_group_b=concept_group_b,
        concept_direction_mode="mean_difference",
    ),
    None,
    0,
)

concept_direction = cd_result.concept_direction
concept_group_a_ids = cd_result.concept_group_a_token_ids
authoritative_yes_id = concept_group_a_ids[0]
concept_group_b_ids = cd_result.concept_group_b_token_ids
authoritative_no_id = concept_group_b_ids[0]

print(f"Concept direction shape: {concept_direction.shape}")
print(f"Direction norm: {concept_direction.norm().item():.4f}")
print(f"Token IDs - Yes: {concept_group_a_ids}, No: {concept_group_b_ids}")

In [ ]:
# Quality gate: project the full vocabulary onto the concept direction
embed_weight = resolve_embedding_weight(module).float()
scores = embed_weight @ concept_direction.to(embed_weight.device)

yes_id = authoritative_yes_id
no_id = authoritative_no_id
k_quality = 15

topk_yes = torch.topk(scores, k_quality)
topk_no = torch.topk(-scores, k_quality)

print(f"Top {k_quality} tokens aligned with the positive direction:")
print(f"{'Rank':<6} {'Token':<20} {'Score':<10}")
print("-" * 36)
for i in range(k_quality):
    token_str = tokenizer.decode([topk_yes.indices[i].item()])
    print(f"{i + 1:<6} {repr(token_str):<20} {topk_yes.values[i].item():<10.4f}")

print(f"\nTop {k_quality} tokens aligned with the negative direction:")
print(f"{'Rank':<6} {'Token':<20} {'Score':<10}")
print("-" * 36)
for i in range(k_quality):
    token_str = tokenizer.decode([topk_no.indices[i].item()])
    print(f"{i + 1:<6} {repr(token_str):<20} {topk_no.values[i].item():<10.4f}")

print(f"\nConcept token scores: 'Yes' = {scores[yes_id].item():+.4f}, 'No' = {scores[no_id].item():+.4f}")
print(f"Score separation: {(scores[yes_id] - scores[no_id]).item():.4f}")

In [ ]:
# Validate the direction on real RTE examples from the dataloader
entailment_mapping = ("Yes", "No")
replacement_model = module.replacement_model
validated_examples = []

print(f"Validating concept direction on {n_validation_batches} batches from test_dataloader:\n")

for batch_idx, batch in enumerate(test_dl):
    if batch_idx >= n_validation_batches:
        break

    labels = batch["labels"]
    label_list = labels.tolist() if isinstance(labels, torch.Tensor) else list(labels)
    input_ids = batch["input"]

    for ex_idx in range(input_ids.shape[0]):
        prompt = tokenizer.decode(input_ids[ex_idx].tolist(), skip_special_tokens=True)
        logits, _ = replacement_model.get_activations(prompt)
        last_logits = logits.squeeze(0)[-1].float().cpu()

        yes_logit = last_logits[yes_id].item()
        no_logit = last_logits[no_id].item()
        gap = yes_logit - no_logit
        predicted_label = 0 if gap > 0 else 1
        true_label = label_list[ex_idx]
        is_correct = predicted_label == true_label

        validated_examples.append(
            {
                "batch_idx": batch_idx,
                "prompt": prompt,
                "label": entailment_mapping[true_label],
                "predicted": entailment_mapping[predicted_label],
                "yes_logit": yes_logit,
                "no_logit": no_logit,
                "gap": gap,
                "is_correct": is_correct,
            }
        )

n_correct = sum(1 for ex in validated_examples if ex["is_correct"])
n_total = len(validated_examples)

print(f"{'Batch':<7} {'Label':<6} {'Pred':<6} {'Yes-No gap':>12} {'Correct':<8} {'Prompt (truncated)'}")
print("-" * 95)
for ex in validated_examples:
    mark = "✓" if ex["is_correct"] else "✗"
    print(
        f"{ex['batch_idx']:<7} {ex['label']:<6} {ex['predicted']:<6} "
        f"{ex['gap']:>+12.4f} {mark:<8} {ex['prompt'][:50]}..."
    )

print(f"\nAccuracy: {n_correct}/{n_total} ({100 * n_correct / n_total:.1f}%)")
correct_examples = [ex for ex in validated_examples if ex["is_correct"]]
print(f"Correct examples available for intervention: {len(correct_examples)}")

---
## Phase B — Attribution & Intervention

Using the validated concept direction, we run the full
intervention_from_concept pipeline on real RTE examples where the model
answered correctly.

This phase demonstrates:
1. Attribution graph generation for real examples from the dataloader
2. Top feature extraction and comparison across multiple graphs
3. Feature intervention with a notebook-safe attribution budget

The notebook intentionally uses the shortest correct prompts from the sampled
batches so the Gemma-2 NNsight circuit-tracer workflow remains executable on a
single GPU.

In [ ]:
# Run intervention_from_concept on the shortest correct examples from the dataloader
pipeline = it.intervention_from_concept
print(f"Pipeline: {' → '.join(op.name for op in pipeline.composition)}")

concept_groups_a = ["▁Yes", "▁True", "▁Correct"]
concept_groups_b = ["▁No", "▁False", "▁Wrong"]
concept_label = "Entailment: Yes − No"

selected = sorted(correct_examples, key=lambda item: len(item["prompt"]))[:2]
print(f"\nSelected {len(selected)} shortest correct examples for intervention:\n")

intervention_results = []
for i, ex in enumerate(selected):
    print(f"--- Example {i + 1}: Expected {ex['label']!r} ---")
    print(f"  Prompt length: {len(ex['prompt'])} characters")
    print(f"  Prompt: {ex['prompt'][:80]}...")

    result = pipeline(
        module,
        AnalysisBatch(
            concept_group_a=concept_groups_a,
            concept_group_b=concept_groups_b,
            concept_label=concept_label,
            concept_direction_mode="paired_rejection",
            prompts=[ex["prompt"]],
        ),
        None,
        0,
        top_n=min(n_top, 5),
        intervention_scale_factor=intervention_scale_factor,
        batch_size=4,
        max_feature_nodes=256,
    )

    pre_logits = result.pre_intervention_logits.float().cpu()
    post_logits = result.post_intervention_logits.float().cpu()
    pre_gap = (pre_logits[yes_id] - pre_logits[no_id]).item()
    post_gap = (post_logits[yes_id] - post_logits[no_id]).item()

    print(f"  Yes−No gap: pre={pre_gap:+.4f}  post={post_gap:+.4f}  change={post_gap - pre_gap:+.4f}")

    intervention_results.append(
        {
            "example": ex,
            "result": result,
            "pre_logits": pre_logits,
            "post_logits": post_logits,
            "pre_gap": pre_gap,
            "post_gap": post_gap,
        }
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nAll {len(intervention_results)} interventions complete.")

In [ ]:
# Display intervention results
key_tokens = [("Yes", yes_id), ("No", no_id)]

for i, ir in enumerate(intervention_results):
    ex = ir["example"]
    print(f"\n{'=' * 60}")
    print(f"Example {i + 1}: Expected {ex['label']!r} | Batch {ex['batch_idx']}")
    display_topk_token_predictions(
        ex["prompt"][:80] + "...",
        ir["pre_logits"],
        ir["post_logits"],
        tokenizer,
        k=5,
        key_tokens=key_tokens,
    )

feature_sets = {}
scores_sets = {}
for i, ir in enumerate(intervention_results):
    label = f"Ex {i + 1}: {ir['example']['label']}"
    result = ir["result"]
    feature_sets[label] = [tuple(f.tolist()) for f in result.top_feature_ids[:5]]
    scores_sets[label] = [s.item() for s in result.top_feature_scores[:5]]

display_top_features_comparison(feature_sets, scores_sets)

print("\nIntervention summary (concept direction amplification):")
print(f"{'Example':<12} {'Label':<6} {'Pre gap':>10} {'Post gap':>10} {'Change':>10}")
print("-" * 52)
for i, ir in enumerate(intervention_results):
    ex = ir["example"]
    delta = ir["post_gap"] - ir["pre_gap"]
    print(f"Ex {i + 1:<9} {ex['label']:<6} {ir['pre_gap']:>+10.4f} {ir['post_gap']:>+10.4f} {delta:>+10.4f}")

In [ ]:
# Cleanup
del module, it_session, session_cfg, datamodule, test_dl
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
else:
    print("Cleanup complete (CPU mode)")

## Summary

This notebook demonstrated end-to-end concept-direction analysis on the RTE task
using real dataloader examples.

| Phase | What | Result |
|-------|------|--------|
| A | Concept direction computation | Embedding-based Yes vs No direction |
| A | Quality gate | Verified token-space separation and concept token scores |
| A | Real-data validation | Checked direction alignment on sampled dataloader batches |
| B | Attribution pipeline | Generated attribution graphs on real RTE examples |
| B | Feature intervention | Amplified concept-aligned features and measured logit shifts |
| B | Cross-example comparison | Compared top features across multiple examples |

The full composite pipeline is:
concept_direction → compute_attribution_graph → graph_node_influence →
extract_top_features → feature_intervention_forward